# 05 — Model Explainability (SHAP)

This notebook uses **SHAP** (SHapley Additive exPlanations) to:
1. Generate **global** feature importance (summary plot)
2. Generate **local** explanation for a single prediction (waterfall / force plot)
3. Save plots to `reports/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import shap

from src.data_loader import load_cleaned_data
from src.features import engineer_features
from src.utils import split_data, get_models_dir, get_reports_dir, setup_logging

setup_logging()
%matplotlib inline

## 1. Load Model & Data

In [ ]:
# Load the trained model
model_path = os.path.join(get_models_dir(), 'model.pkl')
model = joblib.load(model_path)
print(f'Model loaded: {type(model).__name__}')

# Load and prepare data
df = load_cleaned_data()
df = engineer_features(df)
X_train, X_test, y_train, y_test = split_data(df)
print(f'Test set: {X_test.shape}')

## 2. SHAP Explainer

In [ ]:
# Create SHAP explainer
# Use TreeExplainer for tree-based models, otherwise KernelExplainer
model_type = type(model).__name__

if model_type in ['RandomForestClassifier', 'XGBClassifier', 'GradientBoostingClassifier']:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    # For binary classification, TreeExplainer may return a list of 2 arrays
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # positive class
else:
    # Fallback: use KernelExplainer with a sample of training data
    background = shap.sample(X_train, 100)
    explainer = shap.KernelExplainer(model.predict_proba, background)
    shap_values = explainer.shap_values(X_test.iloc[:200])
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

print(f'SHAP values shape: {shap_values.shape}')

## 3. Global Feature Importance (Summary Plot)

In [ ]:
reports_dir = get_reports_dir()

# Summary plot (bar)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'shap_global_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary plot (dot) — shows feature value impact direction
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Feature Impact (Dot Plot)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'shap_dot_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Local Explanation (Single Prediction)

In [ ]:
# Pick a single test instance
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1]
true_label = y_test.iloc[sample_idx]
pred_proba = model.predict_proba(sample)[:, 1][0]

print(f'True label: {true_label}')
print(f'Predicted P(default): {pred_proba:.4f}')
print(f'\nFeature values:')
print(sample.T)

In [ ]:
# Waterfall plot for the single prediction
shap_explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
    data=X_test.iloc[sample_idx].values,
    feature_names=X_test.columns.tolist()
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'SHAP Local Explanation (Sample {sample_idx})', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(reports_dir, 'shap_local_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Force plot for the single prediction
shap.initjs()
force_plot = shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
    shap_values[sample_idx],
    X_test.iloc[sample_idx],
    matplotlib=True,
)
plt.savefig(os.path.join(reports_dir, 'shap_local_force.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Interpretation

### Global (Summary Plot)
- **Repayment delay columns** (`pay_1`, `pay_2`, etc.) dominate feature importance.
- **Delay score** (our engineered feature) also ranks highly.
- This aligns with domain knowledge: past delinquency is the strongest predictor of future default.

### Local (Waterfall / Force Plot)
- The waterfall plot shows how each feature pushes the prediction away from the base rate.
- Red arrows push toward default (higher probability), blue arrows push away.
- This provides **actionable transparency** for loan officers reviewing individual applications.